In [1]:
pip install newsapi-python


  Using cached newsapi_python-0.2.7-py2.py3-none-any.whl (7.9 kB)


You should consider upgrading via the 'c:\Users\KIIT\AppData\Local\Programs\Python\Python38\python.exe -m pip install --upgrade pip' command.


In [7]:
import requests
import pandas as pd
from newsapi import NewsApiClient
from datetime import datetime
import urllib.parse
from datetime import datetime, timedelta  

In [21]:
import pandas as pd
from datetime import datetime, timedelta
import urllib.parse
from newsapi import NewsApiClient

# Function to fetch news based on the provided query, dates, and other parameters
def fetch_news(query, from_date, to_date, language='en', sort_by='relevancy', page_size=100, api_key='YOUR_API_KEY'):
    # Initialize the NewsApiClient with the provided API key
    newsapi = NewsApiClient(api_key=api_key)
    
    # URL encode the query to handle spaces and special characters
    query = urllib.parse.quote(query)  # Correctly encodes spaces as %20 and handles other special characters

    try:
        # Fetch all articles matching the query
        all_articles = newsapi.get_everything(
            q=query,
            from_param=from_date,
            to=to_date,
            language=language,
            sort_by=sort_by,
            page_size=page_size
        )

        # Check if articles were returned
        if all_articles.get('status') == 'ok' and all_articles.get('totalResults', 0) > 0:
            df = pd.DataFrame(all_articles['articles'])  # Extract articles list
            return df
        else:
            print(f"No articles found for query: {query}")
            return pd.DataFrame()  # Return an empty DataFrame if no articles were found

    except Exception as e:
        print(f"An error occurred while fetching news: {e}")
        return pd.DataFrame()

# Get the current time
now = datetime.now()

# Get the time 10 days ago
ten_days_ago = now - timedelta(days=10)

# API Key and query
api_key = "b82276461b68436fa5a5b61891db4625"  # Replace with your actual API key
q = "NVIDIA"  # Simplified query

# Call the fetch_news function with the query and date range
df = fetch_news(query=q, from_date=ten_days_ago, to_date=now, api_key=api_key)

# Check the columns and the first few rows of the DataFrame
print("Columns in the DataFrame:", df.columns)
print("First few rows of the DataFrame:")
print(df.head())

# Drop the 'source' column if it exists
if 'source' in df.columns:
    df_news = df.drop("source", axis=1)
else:
    df_news = df

# Function to preprocess the 'publishedAt' date column
def preprocess_news_date(df):
    # Check if 'publishedAt' exists in the DataFrame
    if 'publishedAt' in df.columns:
        # Convert the 'publishedAt' column to datetime format
        df['publishedAt'] = pd.to_datetime(df['publishedAt'])
        # Extract only the date part (removes time)
        df['publishedAt'] = df['publishedAt'].dt.date
    else:
        print("No 'publishedAt' column found in the data.")
    return df

# Preprocess the news data
df_news = preprocess_news_date(df_news)

# Display the preprocessed DataFrame
print(df_news)

Columns in the DataFrame: Index(['source', 'author', 'title', 'description', 'url', 'urlToImage',
       'publishedAt', 'content'],
      dtype='object')
First few rows of the DataFrame:
                                source     author  \
0  {'id': None, 'name': 'Gizmodo.com'}  Kyle Barr   
1  {'id': None, 'name': 'Gizmodo.com'}  Kyle Barr   
2  {'id': None, 'name': 'Gizmodo.com'}  Kyle Barr   
3  {'id': None, 'name': 'Gizmodo.com'}  Kyle Barr   
4  {'id': None, 'name': 'Gizmodo.com'}  Kyle Barr   

                                               title  \
0  The Nvidia RTX 5080 Is the GPU Gamers Should R...   
1  RTX 5090 Vigilantes Attempt to Fool Scalper Bo...   
2  Nvidia Warns It Will Likely Run Out of RTX 509...   
3  Nvidia GeForce Now on Vision Pro Highlights th...   
4  The Nvidia RTX 5090 Generates So Many Frames, ...   

                                         description  \
0  The Nvidia RTX 5080 cannot match up to the raw...   
1  As Nvidia RTX 5090 GPU is in such hot dema

In [22]:
preprocess_news_df=preprocess_news_date(df_news)
preprocess_news_df.head()

,author,title,description,url,urlToImage,publishedAt,content
0,Kyle Barr,The Nvidia RTX 5080 Is the GPU Gamers Should R...,The Nvidia RTX 5080 cannot match up to the raw...,https://gizmodo.com/nvidia-rtx-5080-review-200...,https://gizmodo.com/app/uploads/2025/01/nVidia...,2025-01-29,"If the $2,000 Nvidia GeForce RTX 5090 is the u..."
1,Kyle Barr,RTX 5090 Vigilantes Attempt to Fool Scalper Bo...,As Nvidia RTX 5090 GPU is in such hot demand m...,https://gizmodo.com/rtx-5090-vigilantes-attemp...,https://gizmodo.com/app/uploads/2025/01/Nvidia...,2025-01-30,"Just as Nvidia warned it would happen, the GeF..."
2,Kyle Barr,Nvidia Warns It Will Likely Run Out of RTX 509...,Some gamers are already camping out during one...,https://gizmodo.com/rtx-5080-stock-shortage-20...,https://gizmodo.com/app/uploads/2025/01/nVidia...,2025-01-29,"Just four years ago, the crypto boom and pande..."
3,Kyle Barr,Nvidia GeForce Now on Vision Pro Highlights th...,"Streaming games with GeForce Now on a $3,500 V...",https://gizmodo.com/geforce-now-on-vision-pro-...,https://gizmodo.com/app/uploads/2025/01/Apple-...,2025-01-31,Ive fallen into the camp of wanting nothing mo...
4,Kyle Barr,"The Nvidia RTX 5090 Generates So Many Frames, ...",Nvidia's multi frame gen on the GeForce RTX 50...,https://gizmodo.com/the-nvidia-rtx-5090-genera...,https://gizmodo.com/app/uploads/2025/01/Nvidia...,2025-01-24,"The $2,000 Nvidia RTX 5090 is so powerful that..."


## Pre Processing the data

In [24]:
def build_prompt(news_df):
    prompt="You are a Financial Analyst tasked with providing insights into recent news articles related to the financial sector. Here are the latest news articles:\n"

    for index, row in news_df.iterrows():
        title=row['title']
        prompt+=f"  **NEWS:** {title}\n"

    prompt+="Please analyse these articles and provide insights into any potential impacts on industry Sentiment and Stock Prices."
    return prompt

# Build a prompt using the news DataFrame
prompt = build_prompt(preprocess_news_df)
print (prompt)


You are a Financial Analyst tasked with providing insights into recent news articles related to the financial sector. Here are the latest news articles:
  **NEWS:** The Nvidia RTX 5080 Is the GPU Gamers Should Really Want
  **NEWS:** RTX 5090 Vigilantes Attempt to Fool Scalper Bots With Fake eBay Listings
  **NEWS:** Nvidia Warns It Will Likely Run Out of RTX 5090 and 5080 Cards at Launch
  **NEWS:** Nvidia GeForce Now on Vision Pro Highlights the Bigger Problem With Apple and Gaming
  **NEWS:** The Nvidia RTX 5090 Generates So Many Frames, It Scares Me
  **NEWS:** Lambda Labs' COO has left the AI cloud provider to head Positron, a startup trying to compete with Nvidia
  **NEWS:** Tech stocks stage partial recovery after market rout sparked by DeepSeek's rise
  **NEWS:** Nvidia stock has crossed a red line that points to more pain after this week's DeepSeek rout
  **NEWS:** DeepSeek is driving demand for Nvidia's H200 chips, some cloud firms say
  **NEWS:** Billionaire investor Ray Dal

In [30]:
pip install --upgrade huggingface_hub transformers

  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.25.1
    Uninstalling huggingface-hub-0.25.1:
      Successfully uninstalled huggingface-hub-0.25.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.45.1
    Uninstalling transformers-4.45.1:
      Successfully uninstalled transformers-4.45.1
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\KIIT\AppData\Local\Programs\Python\Python38\python.exe -m pip install --upgrade pip' command.


In [35]:
from transformers import pipeline

# Load the summarization model from Hugging Face Hub
llm_summarizer = pipeline(
    "summarization",
    model="Falconsai/text_summarization",
    model_kwargs={"temperature": 0.1}
)

c:\Users\KIIT\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
c:\Users\KIIT\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [37]:
from IPython.display import Markdown

In [39]:
from transformers import pipeline
from IPython.display import Markdown

# Initialize the summarizer
llm_summarizer = pipeline(
    "summarization",
    model="Falconsai/text_summarization",
    model_kwargs={"temperature": 0.1}
)

# Get the summary of the prompt
summary = llm_summarizer(prompt)

# Extract the summary text from the list of dictionaries
summary_text = summary[0]['summary_text']

# Display the summary using Markdown
Markdown(summary_text)


c:\Users\KIIT\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
c:\Users\KIIT\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (2536 > 512). Running this sequence through the model will resul

Investors worry DeepSeek reduces AI chip demand, but there's a case for remaining bullish on Nvidia **NEWS:** China’s AI assistant becomes top free iPhone app as US tech stocks take a hit **Nvidia tumbles and tech stocks slide premarket as investors worry about China's AI startup . *NEWS :** DeepSea's ultra-cost-effective AI plummets NVIDIA stock prices — wiping out $500 billion in market valuation**NEWS